# 09 — Anomali Sinyali Keşfi (Ground Truth Kullanmadan)

Kural tabanlı ve istatistiksel sinyaller — ML öncesi feature adayları.

> ML aşamasında `data/ground_truth/labels_30min.csv` ile karşılaştırarak
> precision/recall ölçebilirsin. **Bu notebook'ta açma.**

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name != 'eda':
    ROOT = ROOT / 'eda'
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_all, merge_ue1t_inventory
from utils.plots import setup_style

setup_style()
dfs = load_all()
ue1t = dfs['ue1t_30min'].copy()
inv = dfs['inventory_30min']

In [ ]:
# Feature adayları — 30dk bazında
ue1t['satisiz'] = (ue1t['pompa_satis'] == 0).astype(int)
ue1t['kk_oran'] = np.where(ue1t['pompa_satis'] > 1,
                           ue1t['kayip_kazanc'] / ue1t['pompa_satis'], np.nan)
ue1t['saat'] = ue1t['saat_1'].dt.hour
ue1t['gece'] = ue1t['saat'].between(0, 5).astype(int)

print('Feature özet:')
print(ue1t[['kayip_kazanc','kk_oran','satisiz','gece']].describe())

In [ ]:
# Sinyal 1: Satışsız kayıp (statik sızıntı adayı)
satisiz_kayip = ue1t[(ue1t['pompa_satis']==0) & (ue1t['kayip_kazanc'] < -5)]
print('Satışsız kayıp >5 lt dönem:', len(satisiz_kayip))
top = satisiz_kayip.nsmallest(10, 'kayip_kazanc')
display(top[['saat_1','istasyon_kodu','tank_no','kayip_kazanc','donem_basi_stok','donem_sonu_stok']])

In [ ]:
# Sinyal 2: Gece satışsız düşüş (pompacı manipülasyonu adayı)
gece = ue1t[(ue1t['gece']==1) & (ue1t['pompa_satis']==0) & (ue1t['kayip_kazanc'] < -20)]
print('Gece satışsız düşüş >20 lt:', len(gece))
if len(gece):
    display(gece.nsmallest(5,'kayip_kazanc')[['saat_1','istasyon_kodu','tank_no','kayip_kazanc']])

In [ ]:
# Sinyal 3: Yüksek kk_oran (dinamik sızıntı / decimal adayı)
high = ue1t[(ue1t['pompa_satis'] > 50) & (ue1t['kk_oran'].abs() > 0.05)]
print('|kk_oran| > 5% ve satış >50 lt:', len(high))
display(high.nlargest(10, 'kk_oran', keep='all')[['saat_1','istasyon_kodu','tank_no','pompa_satis','kayip_kazanc','kk_oran']].head(10))

In [ ]:
# Sinyal 4: Seviye donması (şamandıra adayı) — stok değişmeden satış
ue1t_sorted = ue1t.sort_values(['istasyon_kodu','tank_no','saat_1'])
ue1t_sorted['stok_diff'] = ue1t_sorted.groupby(['istasyon_kodu','tank_no'])['donem_sonu_stok'].diff()
stuck = ue1t_sorted[(ue1t_sorted['pompa_satis'] > 30) & (ue1t_sorted['stok_diff'].abs() < 0.01)]
print('Satış var ama stok değişmedi:', len(stuck))
if len(stuck):
    display(stuck.head(8)[['saat_1','istasyon_kodu','tank_no','pompa_satis','donem_sonu_stok','kayip_kazanc']])

In [ ]:
# Sinyal 5: Su sıçraması
inv2 = inv.sort_values(['istasyon_kodu','tank_no','envanter_tarihi']).copy()
inv2['su_diff'] = inv2.groupby(['istasyon_kodu','tank_no'])['su_seviyesi_cm'].diff()
su = inv2[inv2['su_diff'] > 0.05]
print('Su sıçrama:', len(su))

In [ ]:
# Günlük alarm günlerinde 30dk profili
daily = dfs['daily']
alarm_gun = daily[daily['alarm']==1].nlargest(1, 'fark')
if len(alarm_gun):
    row = alarm_gun.iloc[0]
    u = ue1t[(ue1t.istasyon_kodu==row.istasyon_kodu)&(ue1t.tank_no==row.tank_no)
             &(ue1t.saat_1.dt.normalize()==row.tarih)]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(range(len(u)), u['kayip_kazanc'], color='crimson', edgecolor='k')
    ax.axhline(0, color='k')
    ax.set_xlabel('30dk dönem')
    ax.set_ylabel('Kayıp/Kazanç')
    ax.set_title(f"Alarm günü {row.istasyon_kodu} T{row.tank_no} {row.tarih.date()} — fark={row.fark:.1f}")
    plt.tight_layout()
    plt.show()

## Sonuç ve sonraki adımlar

**Keşfedilen sinyaller → feature engineering:**
- `satisiz_kayip`, `kk_oran`, `gece_satisiz_dusus`
- `tx_ue1t_fark`, `su_diff`, `gecikme_dk`
- Manifold çift korelasyonu

**ML aşaması:**
1. Feature'ları birleştir
2. Zaman bazlı train/test split (son 20 gün test)
3. `ground_truth/labels_30min.csv` ile karşılaştır
4. Baseline: SEL alarm vs CatBoost/IsolationForest